# Step 1: Deploy Infrastructure

Deploy the Azure Bicep/ARM template to create the Foundry AIServices resource with model deployment.
Capture outputs and write them to `env/.env`.

## Configuration

In [ ]:
import os
import json
import subprocess
from pathlib import Path

# Get demo root and paths
demo_root = Path("..").resolve()
env_dir = demo_root / "env"
infra_dir = demo_root / "infra"
env_file = env_dir / ".env"

# Hardcoded configuration
RESOURCE_GROUP = "rg-cu01-video"
LOCATION = "uksouth"
DEMO_ID = "cu01"

print(f"Demo root: {demo_root}")
print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location: {LOCATION}")
print(f"Demo ID: {DEMO_ID}")

## Create Resource Group

In [ ]:
print(f"Creating resource group: {RESOURCE_GROUP}")
result = subprocess.run(
    ["az", "group", "create", "--name", RESOURCE_GROUP, "--location", LOCATION],
    capture_output=True,
    text=True,
)

if result.returncode == 0:
    print(f"✓ Resource group created/exists")
else:
    print(f"Error creating resource group:")
    print(result.stderr)
    raise RuntimeError(result.stderr)

## Deploy Bicep Template

In [ ]:
# Get deployer principal ID (current user)
deployer_principal_result = subprocess.run(
    ["az", "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True,
    text=True,
)

if deployer_principal_result.returncode != 0:
    print("Warning: Could not get user principal ID. Deploying without RBAC assignment.")
    deployer_principal_id = ""
else:
    deployer_principal_id = deployer_principal_result.stdout.strip()
    print(f"Deployer principal ID: {deployer_principal_id}")

# Deploy Bicep template
template_file = infra_dir / "main.bicep"
print(f"\nDeploying template: {template_file}")

deploy_command = [
    "az", "deployment", "group", "create",
    "--resource-group", RESOURCE_GROUP,
    "--template-file", str(template_file),
    "--parameters",
    f"location={LOCATION}",
    f"demoId={DEMO_ID}",
]

if deployer_principal_id:
    deploy_command.append(f"deployerPrincipalId={deployer_principal_id}")

result = subprocess.run(deploy_command, capture_output=True, text=True)

if result.returncode == 0:
    print("✓ Deployment successful")
else:
    print("Error deploying template:")
    print(result.stderr)
    raise RuntimeError(result.stderr)

## Extract Outputs

In [ ]:
# Get deployment outputs
print("Extracting deployment outputs...")
outputs_result = subprocess.run(
    ["az", "deployment", "group", "list",
     "--resource-group", RESOURCE_GROUP,
     "--query", "[0].properties.outputs",
     "-o", "json"],
    capture_output=True,
    text=True,
    check=True,
)

outputs = json.loads(outputs_result.stdout)

# Extract values
ai_endpoint = outputs["aiServicesEndpoint"]["value"]
ai_account_name = outputs["aiServicesAccountName"]["value"]
ai_deployment_name = outputs["aiDeploymentName"]["value"]
foundry_project_name = outputs.get("foundryProjectName", {}).get("value", "")
foundry_project_endpoint = outputs.get("foundryProjectEndpoint", {}).get("value", "")
rg_name = outputs["resourceGroupName"]["value"]

print(f"AI Endpoint: {ai_endpoint}")
print(f"AI Account: {ai_account_name}")
print(f"Deployment Name: {ai_deployment_name}")
print(f"Project Name: {foundry_project_name}")
print(f"Project Endpoint: {foundry_project_endpoint}")
print(f"Resource Group: {rg_name}")

## Write Environment File

In [ ]:
# Ensure env directory exists
env_dir.mkdir(exist_ok=True)

# Write .env file
env_content = f"""AZURE_RESOURCE_GROUP={rg_name}
AZURE_AI_ENDPOINT={ai_endpoint}
AZURE_AI_ACCOUNT_NAME={ai_account_name}
CU_MODEL_DEPLOYMENT_NAME={ai_deployment_name}
AZURE_FOUNDRY_PROJECT_NAME={foundry_project_name}
AZURE_FOUNDRY_PROJECT_ENDPOINT={foundry_project_endpoint}
AZURE_AUTH_MODE=aad
"""

with open(env_file, "w") as f:
    f.write(env_content)

print(f"✓ Environment file written to {env_file}")
print(f"\nEnvironment variables:")
print(env_content)

## Verification

✓ Resource group created  
✓ Bicep template deployed  
✓ Outputs extracted  
✓ Environment file written

**Next step:** Run `02_configure.ipynb` to verify the deployment and check connectivity.